# Assignment 4 — Problem Formulation

**Course:** EPA141A Model Based Decision Making — Delft University of Technology  
**Model:** JUSTICE

---

## Learning Outcomes

1. Complete the **XLRM framework** for a climate policy problem under deep uncertainty.
2. Translate **justice principles** into formally specified measurable objectives.
3. Understand the **adaptive policy representation** (EMODPS with RBFs).
4. Explore the design space of possible ECR policies.
5. Formally structure a many-objective optimisation problem and articulate the assumptions embedded in it.

---

## Background

Problem formulation is the first step in structuring a many-objective optimisation problem. Before an MOEA can search for Pareto-optimal policies, we must decide *what* to optimise (objectives), *what* we can control (levers), *what* we cannot control (uncertainties), and *what* model relationships connect them. The **XLRM framework** — eXogenous uncertainties, Levers, Relationships, and Measures of performance — provides a systematic structure for this.

In JUSTICE, climate policies are represented as **adaptive emission control rate (ECR) pathways** generated by **radial basis functions (RBFs)**. Rather than pre-specifying a fixed time-schedule for emission reductions, RBFs map the current climate state (global temperature level and its rate of change) to an emission control rate for each of the 57 RICE50 regions. This **closed-loop** formulation means policies automatically adapt to how the climate actually evolves, rather than following a pre-planned calendar schedule.

The MOEA in Assignment 5 will search over the **244 RBF parameters** (centers, radii, and weights) to find combinations that simultaneously minimise multiple objectives. The choice of **social welfare function** (Utilitarian, Prioritarian, Sufficientarian) embeds normative assumptions that directly shape the solution space.

## Setup — Imports and model configuration

The cell below imports all required packages, configures the EMA Workbench logging level, and initialises the DataLoader and geometric constants (number of regions, timesteps, RBF architecture parameters) that are referenced throughout the assignment.

In [ ]:
import os, sys
# -- Add JUSTICE-main to sys.path so justice internal imports resolve
_here         = os.path.abspath(".")   # CWD = notebook dir in JupyterLab
_justice_root = os.path.normpath(os.path.join(_here, "../JUSTICE-main"))
if _justice_root not in sys.path:
    sys.path.insert(0, _justice_root)
os.chdir(_justice_root)

import warnings; warnings.filterwarnings("ignore")
import importlib, sys, os
import multiprocessing
try:
    multiprocessing.set_start_method("fork")
except RuntimeError:
    pass  # already set

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from ema_workbench import (
    Model, RealParameter, IntegerParameter, CategoricalParameter,
    ScalarOutcome, ArrayOutcome,
    perform_experiments, ema_logging, Policy,
    SequentialEvaluator, MultiprocessingEvaluator,
)
from justice.model import JUSTICE
from justice.util.enumerations import WelfareFunction
from justice.util.data_loader import DataLoader
from justice.objectives.objective_functions import years_above_temperature_threshold

import matplotlib.path as _mpath
def _patched_path_deepcopy(self, memo=None):
    if memo is None: memo = {}
    new_path = _mpath.Path.__new__(_mpath.Path)
    memo[id(self)] = new_path
    verts = self._vertices.copy()
    codes = self._codes.copy() if self._codes is not None else None
    new_path.__init__(verts, codes,
                      _interpolation_steps=self._interpolation_steps, readonly=False)
    return new_path
_mpath.Path.__deepcopy__ = _patched_path_deepcopy

ema_logging.log_to_stderr(ema_logging.INFO)
os.makedirs("../plots", exist_ok=True)

N_REGIONS   = 57
N_TIMESTEPS = 286
N_RBFS      = 4
N_INPUTS    = 2

years   = np.arange(2015, 2301)
tau_hat = np.clip((years - 2015) / (2100 - 2015), 0, 1)
s_curve = 3 * tau_hat**2 - 2 * tau_hat**3

dl = DataLoader()

required = ["justice", "numpy", "pandas", "matplotlib", "ema_workbench", "scipy", "seaborn"]
for pkg in required:
    found = importlib.util.find_spec(pkg) is not None
    print(f"  {chr(10003) if found else chr(10007)}  {pkg}")
print(f"\nPython {sys.version.split()[0]}")
print(f"DataLoader ready -- {len(dl.REGION_LIST)} regions, {N_TIMESTEPS} timesteps")

---

## Step 1 -- Complete the XLRM framework

**Task 1.1** -- Add items to each XLRM category. The pre-filled items are simply suggestions.

**Task 1.2** -- For each item you add, identify its EMA Workbench counterpart:
- Uncertainties (X) -> `RealParameter` or `CategoricalParameter` on `em_model.uncertainties`
- Levers (L) -> `RealParameter` on `em_model.levers`
- Measures (M) -> `ScalarOutcome` on `em_model.outcomes`

This mapping defines the search space and the objective space for the MOEA in Assignment 5.

In [ ]:
# XLRM framework for JUSTICE — fill in each category for your actor perspective.
# Add at least 2–3 items per category and include units/ranges where relevant.
xlrm = {
    "X -- Uncertainties": [
        # Add uncertain parameters here
        # Format: "name: description [lower, upper]"
    ],
    "L -- Levers": [
        # Add policy levers here
        # Include the RBF policy and any other instruments your actor controls
    ],
    "R -- Relationships": [
        "JUSTICE IAM: economy (NEOCLASSICAL) + climate (FaIR) + damage (KALKUHL) + abatement (ENERDATA)",
        "RBF policy: maps temperature state -> per-region emission control rate",
        "RICE50: 57 world regions, 2015-2300 at 1-year timestep",
    ],
    "M -- Measures": [
        # Add outcome metrics your actor cares about
        # Include direction (minimise/maximise) for each
    ],
}

print("XLRM Framework:")
for category, items in xlrm.items():
    print(f"\n{category}")
    for item in items:
        print(f"  - {item}")

## Step 2 -- Justice principles

**Task 2.1** -- Run JUSTICE under BAU (no abatement) with three welfare functions and compare results.

> **Key insight:** What changes is how welfare losses are aggregated, which determines the shape of the objective landscape the MOEA will search. The choice of welfare function therefore changes the Pareto front.

In [ ]:
# Run JUSTICE under BAU (no abatement) for three welfare functions and compare outcomes
welfare_options = {
    "Utilitarian":     WelfareFunction.UTILITARIAN,
    "Prioritarian":    WelfareFunction.PRIORITARIAN,
    "Sufficientarian": WelfareFunction.SUFFICIENTARIAN,
}

bau_results = {}
for wf_name, wf in welfare_options.items():
    JUSTICE.hard_reset()
    model = JUSTICE(
        start_year=2015, end_year=2300, timestep=1,
        scenario=2, climate_ensembles=1, stochastic_run=False,
        social_welfare_function=wf,
    )
    ecr_bau = np.zeros((model.no_of_regions, model.no_of_timesteps))
    model.run(ecr_bau)
    data = model.evaluate()
    bau_results[wf_name] = {
        "welfare":               data["welfare"][0],
        "welfare_loss_damage":   data["welfare_loss_damage"][0],
        "welfare_loss_abatement": data["welfare_loss_abatement"][0],
    }
    print(f"{wf_name}: welfare = {bau_results[wf_name]['welfare']:.4g}")

df_bau = pd.DataFrame(bau_results).T
print("\nBAU outcomes by welfare function:")
print(df_bau.round(4).to_string())

## Step 3 -- Explore the ECR policy design space

**Task 3.1** -- Plot 5 different ECR profiles. This illustrates that the MOEA must search over an enormous space of possible emission pathways, not just a single scalar.

> **Why this matters:** The 244 RBF decision variables can produce any combination of these profiles -- and infinitely many more -- per region per year.

In [ ]:
# Define and plot 5 ECR profiles over the model time horizon
fig, ax = plt.subplots(figsize=(10, 5))

ecr_profiles = {
    "BAU (no abatement)":          np.zeros(N_TIMESTEPS),
    "Low uniform (0.30)":          0.30 * s_curve,
    "High uniform (0.85)":         0.85 * s_curve,
    "Front-loaded (reduce early)": np.clip(0.9 - 0.6 * tau_hat, 0, 1),
    "Back-loaded (delay action)":  np.clip(0.1 + 0.8 * tau_hat**2, 0, 1),
}

for label, ecr in ecr_profiles.items():
    ax.plot(years, ecr, label=label, lw=2)

ax.set_xlabel("Year"); ax.set_ylabel("Emission control rate (0-1)")
ax.set_title("ECR profiles over 2015-2300")
ax.legend(fontsize=9); ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.savefig(os.path.join(_NOTEBOOK_DIR, "plots", "a04_ecr_profiles.png"), dpi=120, bbox_inches="tight")
plt.show()

## Step 4 -- Adaptive policies with RBFs: decision variable structure

**Task 4.1** -- Calculate the exact number of decision variables.

**Task 4.2** -- Explain (in 3-5 sentences) why an RBF adaptive policy is preferable to a fixed time-path ECR schedule when climate sensitivity is uncertain.

*(Write your answer here.)*

In [ ]:
n_rbfs    = N_RBFS      # 4
n_inputs  = N_INPUTS    # 2
n_outputs = N_REGIONS   # 57

# RBF structure: centers + radii + weights
n_centers = n_rbfs * n_inputs
n_radii   = n_rbfs * n_inputs
n_weights = n_outputs * n_rbfs
n_total   = n_centers + n_radii + n_weights

print(f"RBF decision variables:")
print(f"  Centers : {n_rbfs} x {n_inputs} = {n_centers}")
print(f"  Radii   : {n_rbfs} x {n_inputs} = {n_radii}")
print(f"  Weights : {n_outputs} x {n_rbfs} = {n_weights}")
print(f"  TOTAL   : {n_total}")

## RBF adaptive policy

The helper below replicates the RBF formulation used by JUSTICE. Each basis function computes a Gaussian response to the current climate state (scaled temperature and warming rate), and the weighted sum of all basis functions determines the emission control rate prescribed to each of the 57 regions.

The demo parameters below are for illustration only -- centers sampled from [0, 1], radii from [0.05, 0.3]. The actual optimization in Assignment 5 searches centers in [-1, 1] and radii in [0, 1].

In [ ]:
# -------------------------------------------------------------------------
# RBF helper -- matches JUSTICE's original_rbf
# weights shape: (n_rbfs, n_outputs)
# -------------------------------------------------------------------------

def rbf_policy(state_vector, centers, radii, weights):
    """
    state_vector : (n_inputs,)
    centers      : (n_rbfs, n_inputs)
    radii        : (n_rbfs, n_inputs)
    weights      : (n_rbfs, n_outputs)
    returns      : (n_outputs,)  ECR clipped to [0, 1]
    """
    phi = np.exp(-np.sum(((state_vector - centers) / (radii + 1e-6))**2, axis=1))
    weighted = weights * phi[:, np.newaxis]   # (n_rbfs, n_outputs)
    return np.clip(weighted.sum(axis=0), 0.0, 1.0)


# Demo parameters
np.random.seed(42)
demo_centers = np.random.uniform(0.0, 1.0, (N_RBFS, N_INPUTS))
demo_radii   = np.random.uniform(0.05, 0.3,  (N_RBFS, N_INPUTS))
demo_weights = np.random.uniform(0.0,  1.0,  (N_RBFS, N_REGIONS))

# JUSTICE scaling constants
T_MIN, T_MAX = 0.0, 16.0
D_MIN, D_MAX = 0.0,  2.0

# Synthetic SSP-like temperature trajectories
def smooth_rise(t_norm, t_end, t_start=1.0):
    return t_start + (t_end - t_start) * (3*t_norm**2 - 2*t_norm**3)

scenarios = {
    "Low  (SSP1-1.9)":    (smooth_rise(tau_hat, 1.6),  "#1E88E5"),
    "Medium (SSP2-4.5)":  (smooth_rise(tau_hat, 3.2),  "#FF9800"),
    "High (SSP5-8.5)":    (smooth_rise(tau_hat, 5.8),  "#E53935"),
}

def compute_difference_signal(temp_traj):
    diff  = np.zeros(N_TIMESTEPS)
    prev  = 0.0
    for t in range(N_TIMESTEPS):
        if t % 5 == 0:
            diff[t] = temp_traj[t] - prev
            prev     = temp_traj[t]
        else:
            diff[t]  = diff[t-1] if t > 0 else 0.0
    return diff

scenario_data = {}
for name, (temp_traj, color) in scenarios.items():
    diff_traj  = compute_difference_signal(temp_traj)
    sc_temp    = (temp_traj - T_MIN) / (T_MAX - T_MIN)
    sc_diff    = np.clip((diff_traj - D_MIN) / (D_MAX - D_MIN), 0, 1)
    ecr_all    = np.array([
        rbf_policy(np.array([sc_temp[t], sc_diff[t]]), demo_centers, demo_radii, demo_weights)
        for t in range(N_TIMESTEPS)
    ])
    scenario_data[name] = dict(
        sc_temp=sc_temp, sc_diff=sc_diff,
        ecr_mean=ecr_all.mean(axis=1),
        ecr_p10=np.percentile(ecr_all, 10, axis=1),
        ecr_p90=np.percentile(ecr_all, 90, axis=1),
        color=color,
    )

# Decision surface
n_grid   = 80
t_axis   = np.linspace(0, 1, n_grid)
d_axis   = np.linspace(0, 1, n_grid)
TG, DG   = np.meshgrid(t_axis, d_axis)

ecr_surface = np.zeros((n_grid, n_grid))
for i in range(n_grid):
    for j in range(n_grid):
        ecr_surface[i, j] = rbf_policy(
            np.array([TG[i, j], DG[i, j]]), demo_centers, demo_radii, demo_weights
        ).mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
cf = ax.contourf(t_axis, d_axis, ecr_surface, levels=25, cmap="RdYlGn", vmin=0, vmax=1)
cs = ax.contour(t_axis, d_axis, ecr_surface, levels=[0.2, 0.4, 0.6, 0.8],
                colors="black", linewidths=0.7, alpha=0.4)
ax.clabel(cs, fmt="%.1f", fontsize=8)
plt.colorbar(cf, ax=ax).set_label("Mean ECR across 57 regions", fontsize=9)
for name, d in scenario_data.items():
    ax.plot(d["sc_temp"], d["sc_diff"], color=d["color"], linewidth=1.8, label=name, alpha=0.9)
ax.set_xlabel("Scaled temperature T/16 [0->1]", fontsize=10)
ax.set_ylabel("Scaled warming rate dT_5yr/2 [0->1]", fontsize=10)
ax.set_title("(A) RBF decision surface in JUSTICE state space", fontsize=10)
ax.legend(fontsize=8, loc="upper left")

ax = axes[1]
for name, d in scenario_data.items():
    ax.fill_between(d["sc_temp"], d["ecr_p10"], d["ecr_p90"], alpha=0.15, color=d["color"])
    ax.plot(d["sc_temp"], d["ecr_mean"], color=d["color"], linewidth=2.0, label=name)
ax.set_xlabel("Scaled temperature T/16 [0->1]", fontsize=10)
ax.set_ylabel("Emission Control Rate (ECR)", fontsize=11)
ax.set_title("(B) State -> ECR (no year axis)", fontsize=10)
ax.legend(fontsize=8)
ax.set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.savefig("../plots/a04ema_rbf_policy_visual.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: a04ema_rbf_policy_visual.png")

### Panel A -- the decision surface

The 2D grid shows every possible combination of climate state the RBF could encounter. X-axis: scaled global temperature. Y-axis: scaled 5-year warming rate. The colour at each point is the mean ECR the demo RBF would prescribe.

### Panel B -- state to ECR

This shows how ECR evolves as temperature rises along each SSP path. Unlike a fixed schedule, the RBF never looks at the year -- it only reads the current climate state.

## Step 5 -- Define the multi-objective optimisation problem

This step translates the XLRM framework and RBF policy structure into a concrete optimisation configuration. You will make four decisions:

1. **Social welfare function** -- whose preferences does the model represent?
2. **Reference SSP-RCP scenario** -- under what emissions pathway do you optimise?
3. **FaIR ensemble strategy** -- how do you represent deep climate uncertainty efficiently?
4. **Objectives and epsilon-resolution** -- what do you optimise, in which direction, and how finely?

The result is saved as `config/config_student.json`. Assignment 5 loads this file to configure the MOEA run.

### Task 5.1 -- Choose a social welfare function

JUSTICE supports multiple social welfare functions (you explored them in Step 2). The MOEA will optimise the 244 RBF parameters to minimise welfare loss as measured by your chosen function.

**Question:** Which welfare function should the reference optimisation use, and why? Justify your choice in 3-5 sentences.

*(Write your answer here.)*

In [ ]:
# Choose the welfare function for your optimisation run.
# Justify your choice in the reflection questions.
# Options: WelfareFunction.UTILITARIAN, WelfareFunction.PRIORITARIAN,
#          WelfareFunction.SUFFICIENTARIAN

WELFARE_OPTIONS = {
    "Utilitarian": WelfareFunction.UTILITARIAN,
    # Add more if you want to compare welfare functions
}

print("Welfare functions for optimisation:")
for name, wf in WELFARE_OPTIONS.items():
    print(f"  {name}: {wf}")

### Task 5.2 -- Choose the reference SSP-RCP scenario

The MOEA evaluates each candidate policy under a **reference scenario**. JUSTICE supports eight SSP-RCP pathways (indices 0-7).

**Question:** Which scenario should the reference optimisation use, and why?

*(Write your answer here.)*

In [ ]:
# SSP-RCP scenarios supported by JUSTICE (indices 0-7)
ssp_rcp_labels = {
    0: "SSP1-1.9  (strong mitigation)",
    1: "SSP1-2.6",
    2: "SSP2-4.5  (middle of the road)",
    3: "SSP3-7.0",
    4: "SSP4-3.4",
    5: "SSP4-6.0",
    6: "SSP5-3.4-OS",
    7: "SSP5-8.5  (high emissions)",
}

print("Available SSP-RCP scenarios:")
for idx, label in ssp_rcp_labels.items():
    print(f"  {idx}  {label}")

# Choose a scenario index — justify your choice in the reflection questions
CHOSEN_SCENARIO_INDEX = 2   # SSP2-4.5 (middle of the road) is a common reference
print(f"\nChosen: index {CHOSEN_SCENARIO_INDEX}  ({ssp_rcp_labels[CHOSEN_SCENARIO_INDEX]})")

### Task 5.3 -- Select FaIR climate ensemble members

JUSTICE runs with 1001 FaIR ensemble members, each representing a different climate system configuration. Using all 1001 per function evaluation is accurate but slow.

**Important:** FaIR ensemble indices are **not ordered by ECS** -- they are identifiers, not a ranking.

**Question:** How many ensemble members should a local optimisation run use, and how should they be selected?

*(Write your answer here.)*

In [ ]:
FULL_ENSEMBLE_SIZE = 1001
N_ENSEMBLE_LOCAL   = 10   # balance between diversity and runtime

# Select N_ENSEMBLE_LOCAL indices spread uniformly across the full ensemble
ensemble_indices = list(np.linspace(1, 1000, N_ENSEMBLE_LOCAL, dtype=int))

print(f"Full FaIR ensemble : {FULL_ENSEMBLE_SIZE} members")
print(f"Local subset       : {N_ENSEMBLE_LOCAL} members")
print(f"Selected indices   : {ensemble_indices}")

# Verify the model loads with this subset
JUSTICE.hard_reset()
test_model = JUSTICE(start_year=2015, end_year=2100, timestep=5,
                     scenario=2, climate_ensembles=ensemble_indices[0], stochastic_run=False,
                     social_welfare_function=WelfareFunction.UTILITARIAN)
print(f"\nModel loaded: {test_model.no_of_regions} regions, {test_model.no_of_timesteps} timesteps")

### Task 5.4 -- Define the objectives and epsilon-resolution

JUSTICE returns four scalar outputs per policy evaluation. You must decide:
- Which to include as **optimisation objectives**
- The **direction** (minimise or maximise)
- The **epsilon value** -- the Pareto archive resolution; smaller epsilon = finer front = more NFE required

| Output | Typical scale | Meaning |
|---|---|---|
| `welfare` | ~10^2-10^5 | Absolute welfare loss; lower = better |
| `fraction_above_threshold` | 0-1 | Fraction of FaIR members where global T > 2 C in 2100; lower = better |
| `welfare_loss_damage` | ~10^3-10^4 | Welfare burden from climate damages |
| `welfare_loss_abatement` | ~10^3-10^4 | Welfare burden from abatement costs |

**Question:** What direction and epsilon should each objective have? Why are the epsilon values different across objectives?

*(Write your answer here.)*

In [ ]:
from ema_workbench import ScalarOutcome

# Define four objectives with direction (MINIMIZE or MAXIMIZE) and epsilon values.
# Epsilon controls Pareto archive granularity — a good starting point is ~1% of
# the objective scale. Use your BAU outcome values above to calibrate.

objectives = [
    ScalarOutcome("welfare",                  kind=ScalarOutcome.MINIMIZE, epsilon=...),
    ScalarOutcome("fraction_above_threshold", kind=ScalarOutcome.MINIMIZE, epsilon=...),
    ScalarOutcome("welfare_loss_damage",      kind=ScalarOutcome.MAXIMIZE, epsilon=...),
    ScalarOutcome("welfare_loss_abatement",   kind=ScalarOutcome.MAXIMIZE, epsilon=...),
]

print("Objectives:")
for o in objectives:
    print(f"  {o.name:35s}  kind={o.kind}  epsilon={o.epsilon}")

### Task 5.5 -- Assemble and save the optimisation configuration

Collect all four decisions into a single JSON config file. `run_optimization_local.py` (Assignment 5) reads this file to configure every aspect of the MOEA run.

In [ ]:
import json, os

_NOTEBOOK_DIR = os.path.abspath(".")
_CONFIG_DIR   = os.path.join(_NOTEBOOK_DIR, "../config")
os.makedirs(_CONFIG_DIR, exist_ok=True)

# Assemble the optimisation configuration from your choices above
student_config = {
    "start_year":                        2015,
    "end_year":                          2300,
    "data_timestep":                     5,
    "timestep":                          1,
    "emission_control_start_year":       2020,
    "n_rbfs":                            N_RBFS,
    "n_inputs":                          N_INPUTS,
    "epsilons":                          [o.epsilon for o in objectives],
    "welfare_function":                  list(WELFARE_OPTIONS.keys())[0],
    "n_ensemble":                        N_ENSEMBLE_LOCAL,
    "ensemble_indices":                  ensemble_indices,
    "reference_ssp_rcp_scenario_index":  CHOSEN_SCENARIO_INDEX,
    "temperature_year_of_interest":      2100,
}

config_path = os.path.join(_CONFIG_DIR, "config_student.json")
with open(config_path, "w") as f:
    json.dump(student_config, f, indent=2)

print(f"Config saved: {config_path}")
print(json.dumps(student_config, indent=2))

---
## Reflection Questions — Model Answers

**1. Single-scenario optimisation.** The MOEA optimises under SSP2-RCP4.5 alone. What risk does this introduce?


*(Write your answer here.)*


**2. Closed-loop vs. open-loop policies.** The RBF encodes a *decision rule* (react to current climate state) rather than a *decision schedule* (follow a fixed time path). Why is the closed-loop formulation preferable when climate sensitivity is uncertain? What is the computational cost of this flexibility?


*(Write your answer here.)*


**3. FaIR ensemble trade-off.** Using 10 instead of 1001 ensemble members reduces runtime ~100×. What does it sacrifice?


*(Write your answer here.)*